# DuckLake na prática — arquivos, catálogo, dlt e dbt

Mesma arquitetura da Strattum, mas o storage é **DuckLake** em vez do `.duckdb`:

```
fonte (SQLite) --dlt--> RAW (lake.raw.orders) --dbt--> CLEAN (lake.main_clean.orders_clean)
                 destination=ducklake            +database='lake' (materializa NATIVO)
```

**DuckLake = Parquet no storage (pasta `lake/`) + metadados num catálogo SQL (`catalog.ducklake`).**
Este notebook mostra os dois: os **arquivos** gerados e o **catálogo** por dentro.

> A lógica reutilizável vive em [`ducklake_demo.py`](ducklake_demo.py); aqui a gente chama passo a passo pra inspecionar.

In [ ]:
import sys
sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes/05-formato-storage-lake")
import ducklake_demo as d   # ao importar: define paths e escreve o profiles.yml do dbt
import dlt
from dlt.sources.sql_database import sql_database
from dlt.destinations import ducklake

print("catálogo:", d.CATALOG)
print("storage :", d.LAKE_DIR)
d.reset(); d.seed(100)   # começa limpo + 100 pedidos na fonte
print("fonte semeada: 100 orders")

## 1) dlt → RAW (destination=ducklake)

**Como usar no dlt:** o destino é o `ducklake(...)` — catálogo + storage. O dlt escreve
Parquet no `lake/` e registra tudo no catálogo. `write_disposition="replace"` = full load.

In [ ]:
dest = ducklake(credentials=dict(
    ducklake_name="lake",
    catalog=f"duckdb:///{d.CATALOG}",        # metadados (em prod: Postgres)
    storage={"bucket_url": d.LAKE_DIR},        # dados Parquet (em prod: MinIO/S3)
))
pipe = dlt.pipeline("ducklake_demo", destination=dest,
                    dataset_name="raw", pipelines_dir=f"{d.ART}/dlt")

src = sql_database(credentials=d.SRC, table_names=["orders"], backend="sqlalchemy")
pipe.run(src, write_disposition="replace")
print("RAW.orders =", d.count("orders", "raw"))

### Ver os arquivos e o catálogo gerados

`show_files` lista os Parquet no storage; `show_catalog` abre o catálogo e mostra as tabelas de
metadados do DuckLake — repare em `ducklake_snapshot` (versões), `ducklake_table` e
`ducklake_data_file` (cada Parquet registrado, com `record_count`).

In [ ]:
d.show_files("após dlt RAW")
d.show_catalog("após dlt RAW")

## 2) dbt → CLEAN (materializa NATIVO no DuckLake)

**Como consumir no dbt:** o profile usa DuckDB em memória (`path: ":memory:"`, só engine) e faz
`attach` do catálogo DuckLake (alias `lake`). O model tem `database='lake'` → o dbt materializa
**direto no lake**, sem ponte e sem write duplo. Abaixo o profile e o model reais:

In [ ]:
print("=== dbt-ducklake/profiles.yml ===")
print(open(f"{d.DBT_DIR}/profiles.yml").read())
print("=== models/orders_clean.sql ===")
print(open(f"{d.DBT_DIR}/models/orders_clean.sql").read())

In [ ]:
d.dbt_run(full_refresh=True)   # --full-refresh = overwrite; reconstrói a CLEAN no lake
print("CLEAN.orders_clean =", d.count("orders_clean", "main_clean"))
d.show_files("após dbt CLEAN")   # repare no novo lake/main_clean/orders_clean/*.parquet

## 3) Incremental (dlt merge + dbt incremental) → snapshots = versões

Chegam +20 pedidos. O dlt traz só o delta (`merge` + cursor `updated_at`); o dbt processa só as
linhas novas. Cada escrita vira um **snapshot** novo no catálogo (time travel).

In [ ]:
d.seed(20, start=100, day="2026-02-01")
src_inc = sql_database(credentials=d.SRC, table_names=["orders"], backend="sqlalchemy")
src_inc.orders.apply_hints(incremental=dlt.sources.incremental("updated_at"))
pipe.run(src_inc, write_disposition="merge", primary_key="id")
d.dbt_run(full_refresh=False)
print("RAW =", d.count("orders", "raw"), "| CLEAN =", d.count("orders_clean", "main_clean"), "(esperado 120)")
d.show_files("após incremental")
d.show_catalog("após incremental — repare no nº de snapshots (versões)")

## 4) Consumir no DuckDB (qualquer engine que fale DuckLake)

Basta `ATTACH 'ducklake:<catálogo>'` e consultar como tabela normal. O email saiu normalizado
(`lower/trim`) pela transform do dbt.

In [ ]:
import duckdb
con = duckdb.connect(); con.execute("INSTALL ducklake; LOAD ducklake")
con.execute(f"ATTACH 'ducklake:{d.CATALOG}' AS lake (READ_ONLY)")
print("amostra clean:", con.execute(
    "SELECT id, amount, email FROM lake.main_clean.orders_clean ORDER BY id LIMIT 5").fetchall())
print("linhas clean:", con.execute("SELECT count(*) FROM lake.main_clean.orders_clean").fetchone()[0])
con.close()

## O que isso mostra

- **Dados = Parquet aberto** em `lake/` (`raw/orders/`, `main_clean/orders_clean/`), **sem** o
  arquivo `.duckdb` monolítico.
- **Metadados = catálogo SQL** (`catalog.ducklake`): `ducklake_table`, `ducklake_data_file`
  (cada Parquet + `record_count`), `ducklake_snapshot` (versões).
- **dlt** escreve no lake nativamente (`destination=ducklake`), com `replace` e `merge`/incremental.
- **dbt** materializa a CLEAN **direto no lake** (`+database='lake'`) — sem ponte, sem write duplo
  (diferente do Delta, onde o dbt-duckdb não escreve nativo).
- **DuckDB é só a engine**; o storage é aberto → destrava MinIO e dá snapshot isolation.